Linear Regression and variance inflation factor

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn import metrics
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split

In [2]:
#Import smallest dataset to be appended to other datasets
#This dataset is measure of a few simple totals that work as a representation of the logistical complexity of each airport
domestic_data_2024 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\T100_Domestic_Market_and_Segment_Data_8942359590531559889.csv")
domestic_data_2024.drop(["year", "enplanements", "arrivals", "OBJECTID"], axis=1, inplace=True) #Redundant with other columns
domestic_data_2024.head()

,origin,passengers,departures,freight,mail
0,01A,17,5,0,0
1,05A,1,1,0,0
2,06A,55,67,139,0
3,09A,43,15,0,0
4,1B1,32,7,0,0


In [3]:
#Import first dataset, 2015 Flight Delay Data
#Columns 7,8 needed dtype specified directly as infer failed

flights_2015 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2015_Delay_Data\flights.csv", dtype={"DESTINATION_AIRPORT":str, "ORIGIN_AIRPORT":str})

flights_2015.fillna({"AIR_SYSTEM_DELAY":0,"SECURITY_DELAY":0,"AIRLINE_DELAY":0,"LATE_AIRCRAFT_DELAY":0,"WEATHER_DELAY":0}, inplace=True)
flights_2015.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0


In [4]:
#Append airport logistics numbers to the 2015 Flight Delay Dataset
domestic_data_2024.rename({"origin" : "ORIGIN_AIRPORT"}, inplace=True, axis=1)
flight_2015_extended = pd.merge(flights_2015, domestic_data_2024, how='left', on="ORIGIN_AIRPORT")
flight_2015_extended.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,passengers,departures,freight,mail
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,NaN,0.0,0.0,0.0,0.0,0.0,2702278.0,71765.0,3.116064e+09,95095127.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,NaN,0.0,0.0,0.0,0.0,0.0,17666714.0,138571.0,2.182348e+08,8795256.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,NaN,0.0,0.0,0.0,0.0,0.0,22288303.0,191107.0,3.490318e+08,14452644.0


In [5]:
#To perform any linear regression we need to drop non-numeric columns, columns that breakdown the delay into reasons, and unhelpful columns such as Year
#The non-numeric columns need dropped due to the number of unique values and the lack of meaningful directionality of those values
#Also need to drop Elapsed Time which is Air Time plus Taxiing time at both airports as they directly calculate the delay

flight_2015_delay_vars = flight_2015_extended.drop(["YEAR", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DIVERTED", "ELAPSED_TIME", "AIR_TIME", "TAXI_IN", "TAXI_OUT",
                                                          "CANCELLED", "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", 
                                                          "WEATHER_DELAY"], axis=1)
flight_2015_delay_vars.dropna(axis=0, inplace=True)

target = flight_2015_delay_vars["ARRIVAL_DELAY"]
variables = flight_2015_delay_vars.drop("ARRIVAL_DELAY", axis=1)

In [6]:
#Scale variables
variables_scl = StandardScaler().fit_transform(variables)

lr_model = LinearRegression()
lr_model.fit(variables_scl,target)
preds = lr_model.predict(variables_scl)
rmse = metrics.root_mean_squared_error(target, preds)

print(f"The RMSE of the model with simple Linear Regression was {rmse:.4f} minutes")

#Create 2nd-degree terms
variables_poly = PolynomialFeatures(degree=2, include_bias=False).fit_transform(variables_scl)

X_train, X_test, y_train, y_test = train_test_split(variables_poly, target, random_state=42)

lr_model_poly = LinearRegression().fit(X_train,y_train)

train_preds = lr_model_poly.predict(X_train)
test_preds = lr_model_poly.predict(X_test)

rmse_poly_train = metrics.root_mean_squared_error(y_train, train_preds)
rmse_poly_test = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE of the model using 2nd degree terms was {rmse_poly_train:.4f} minutes on the training data and {rmse_poly_test:.4f} minutes on the testing data")

The RMSE of the model with simple Linear Regression was 12.6370 minutes
The RMSE of the model using 2nd degree terms was 9.3138 minutes on the training data and 9.3017 minutes on the testing data


In [ ]:
print("Variables\t\tWeights")
print("-"*30)

for v,c in zip(variables.columns, lr_model.coef_):
    print(f"{v:<20}\t{c:.6f}")

Variables		Weights
------------------------------
MONTH               	-0.581533
DAY                 	-0.119335
DAY_OF_WEEK         	-0.258938
SCHEDULED_DEPARTURE 	-0.916800
DEPARTURE_TIME      	-2.646021
DEPARTURE_DELAY     	37.715344
WHEELS_OFF          	3.360734
SCHEDULED_TIME      	-11.905356
DISTANCE            	9.952193
WHEELS_ON           	0.639490
SCHEDULED_ARRIVAL   	-0.665037
ARRIVAL_TIME        	0.237952
passengers          	-2.263236
departures          	1.896745
freight             	-0.028665
mail                	-0.011430


In [10]:
print("Linear Regression on Standardized")
print("\t  2015 Features")
print("Variables\t\tWeights")
print("-"*30)

for v,c in zip(variables.columns, lr_model.coef_):
    print(f"{v:<20}\t{c:.4f}")

Linear Regression on Standardized
	  2015 Features
Variables		Weights
------------------------------
MONTH               	-0.5815
DAY                 	-0.1193
DAY_OF_WEEK         	-0.2589
SCHEDULED_DEPARTURE 	-0.9168
DEPARTURE_TIME      	-2.6460
DEPARTURE_DELAY     	37.7153
WHEELS_OFF          	3.3607
SCHEDULED_TIME      	-11.9054
DISTANCE            	9.9522
WHEELS_ON           	0.6395
SCHEDULED_ARRIVAL   	-0.6650
ARRIVAL_TIME        	0.2380
passengers          	-2.2632
departures          	1.8967
freight             	-0.0287
mail                	-0.0114


Simply trying to use a linear regression model to predict flight delays like this is not feasible and not the ultimate goal of the models I will be building for this project. The model is dominated by the departure delay. This is a reasonable estimate for most flights because most flights are approximately on time. The RMSE of 12.64 minutes is going to serve as the baseline for other models.

In [28]:
print("Variables\t\tIVF")
print("-"*30)

for i in range(len(variables.columns)):
    vif = variance_inflation_factor(variables, i)
    print(f"{variables.columns[i]:<20}\t{vif:.6f}")

Variables		IVF
------------------------------
MONTH               	0.898604
DAY                 	0.906024
DAY_OF_WEEK         	0.887124
SCHEDULED_DEPARTURE 	14.572378
DEPARTURE_TIME      	30.953868
DEPARTURE_DELAY     	1.092674
WHEELS_OFF          	18.849893
SCHEDULED_TIME      	23.998338
DISTANCE            	25.662224
WHEELS_ON           	17.722413
SCHEDULED_ARRIVAL   	4.885244
ARRIVAL_TIME        	16.343043
passengers          	28.640584
departures          	29.680271
freight             	2.657939
mail                	1.892845


There seem to be some highly correlated variables. There are some natural correlations between departure time and arrival time, passengers and departures, schedulded departure and departure delay, etc. If ultimately I choose to go with logistic regression as a means of predicting significant delays then the multicollinearity of the dataset might be an issue to address.

In [21]:
#Import 2nd Dataset, 2019 Flight Delay Info

delays_2019 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2019_Delay_Data\full_data_flightdelay.csv")

delays_2019.head()

,MONTH,DAY_OF_WEEK,DEP_DEL15,DEP_TIME_BLK,DISTANCE_GROUP,SEGMENT_NUMBER,CONCURRENT_FLIGHTS,NUMBER_OF_SEATS,CARRIER_NAME,AIRPORT_FLIGHTS_MONTH,...,PLANE_AGE,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,PRCP,SNOW,SNWD,TMAX,AWND
0,1,7,0,0800-0859,2,1,25,143,Southwest Airlines Co.,13056,...,8,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
1,1,7,0,0700-0759,7,1,29,191,Delta Air Lines Inc.,13056,...,3,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
2,1,7,0,0600-0659,7,1,27,199,Delta Air Lines Inc.,13056,...,18,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
3,1,7,0,0600-0659,9,1,27,180,Delta Air Lines Inc.,13056,...,2,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
4,1,7,0,0001-0559,7,1,10,182,Spirit Air Lines,13056,...,1,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91


In [22]:
#The target variable for this dataset is "DEP_DEL15" it is a binary 1 if the flight was delayed more than 15 minutes in its departure delay else 0
#Predicting this with a linear model is inappropriate, but I will do so anyways for the excercise
num_delays_2019 = delays_2019.drop(["DEP_TIME_BLK", "CARRIER_NAME", "DEPARTING_AIRPORT", "PREVIOUS_AIRPORT"], axis = 1)

target_2019 = num_delays_2019["DEP_DEL15"]
variables_2019 = num_delays_2019.drop(["DEP_DEL15"], axis = 1)

In [28]:
#Scale variables
variables_scl = StandardScaler().fit_transform(variables_2019)

lr_model = LinearRegression()
lr_model.fit(variables_scl,target_2019)
preds = lr_model.predict(variables_scl)
rmse = metrics.root_mean_squared_error(target_2019, preds)

print(f"The RMSE of the model with simple Linear Regression was {rmse:.4f}")

#Create 2nd-degree terms
variables_poly = PolynomialFeatures(degree=2, include_bias=False).fit_transform(variables_scl)

X_train, X_test, y_train, y_test = train_test_split(variables_poly, target_2019, random_state=42)

lr_model_poly = LinearRegression().fit(X_train,y_train)

train_preds = lr_model_poly.predict(X_train)
test_preds = lr_model_poly.predict(X_test)

rmse_poly_train = metrics.root_mean_squared_error(y_train, train_preds)
rmse_poly_test = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE of the model using 2nd degree terms was {rmse_poly_train:.4f} minutes on the training data and {rmse_poly_test:.4f} minutes on the testing data")

The RMSE of the model with simple Linear Regression was 0.3856
The RMSE of the model using 2nd degree terms was 0.3812 minutes on the training data and 0.3814 minutes on the testing data


In [22]:
print("Variables\t\t\t    Weights")
print("-"*50)

for v,c in zip(variables_2019.columns, lr_model.coef_):
    print(f"{v:<35}{c:.6f}")

Variables			    Weights
--------------------------------------------------
MONTH                              -0.002082
DAY_OF_WEEK                        0.000279
DISTANCE_GROUP                     0.006980
SEGMENT_NUMBER                     0.030335
CONCURRENT_FLIGHTS                 -0.000528
NUMBER_OF_SEATS                    0.000477
AIRPORT_FLIGHTS_MONTH              0.000003
AIRLINE_FLIGHTS_MONTH              0.000001
AIRLINE_AIRPORT_FLIGHTS_MONTH      -0.000001
AVG_MONTHLY_PASS_AIRPORT           -0.000000
AVG_MONTHLY_PASS_AIRLINE           -0.000000
FLT_ATTENDANTS_PER_PASS            0.000000
GROUND_SERV_PER_PASS               -0.000000
PLANE_AGE                          0.000753
LATITUDE                           -0.000406
LONGITUDE                          0.000909
PRCP                               0.083839
SNOW                               0.048607
SNWD                               0.010390
TMAX                               0.000149
AWND                               0.0

An RMSE of .3856. That is not a great result for predicting a binary. It is mostly a reflection of selecting an inappropriate model for this circumstances.

In [24]:
print("Variables\t\t\t     IVF")
print("-"*50)

for i in range(len(variables_2019.columns)):
    vif = variance_inflation_factor(variables_2019, i)
    print(f"{variables_2019.columns[i]:<30}\t{vif:.6f}")

Variables			     IVF
--------------------------------------------------
MONTH                         	5.182864
DAY_OF_WEEK                   	4.788773
DISTANCE_GROUP                	4.996590
SEGMENT_NUMBER                	4.380821
CONCURRENT_FLIGHTS            	9.935791
NUMBER_OF_SEATS               	25.382543
AIRPORT_FLIGHTS_MONTH         	71.250372
AIRLINE_FLIGHTS_MONTH         	54.469134
AIRLINE_AIRPORT_FLIGHTS_MONTH 	3.462027
AVG_MONTHLY_PASS_AIRPORT      	57.883071
AVG_MONTHLY_PASS_AIRLINE      	56.352546
FLT_ATTENDANTS_PER_PASS       	3.156517
GROUND_SERV_PER_PASS          	17.881470
PLANE_AGE                     	4.667828
LATITUDE                      	29.232871
LONGITUDE                     	31.205532
PRCP                          	1.115833
SNOW                          	1.127314
SNWD                          	1.179674
TMAX                          	16.654658
AWND                          	6.407003


In [6]:
#Import 3rd Dataset

delay_causes = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\Flight_Delay_and_Causes_Data\Flight_delay.csv")
#Remove the few duplicate datapoints
delay_causes.drop_duplicates(keep="first", inplace=True)
#Drop columns with no information (all values are the same)
delay_causes.drop(["Cancelled","Diverted","CancellationCode"],axis=1, inplace=True)
#Fix long delays to break out of 24 hour time to show the real delay length
data_to_shift = delay_causes[delay_causes["ArrTime"] < (delay_causes["CRSArrTime"] - 100)].index
delay_causes.loc[data_to_shift,"ArrTime"] = delay_causes.loc[data_to_shift,"ArrTime"] + 2400
                           
delay_causes.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,Dest,Dest_Airport,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,BWI,Baltimore-Washington International Airport,515,3,10,2,0,0,0,32
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,LAS,McCarran International Airport,1591,3,7,10,0,0,0,47
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,MCO,Orlando International Airport,828,6,8,8,0,0,0,72
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,PHX,Phoenix Sky Harbor International Airport,1489,7,8,3,0,0,0,12
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,TPA,Tampa International Airport,838,4,9,0,0,0,0,16


In [7]:
#Append Logistics Dataset
domestic_data_2024.rename({"ORIGIN_AIRPORT" : "Origin"}, inplace=True, axis=1)
delay_causes_extended = pd.merge(delay_causes, domestic_data_2024, how='left', on="Origin")
delay_causes_extended.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,passengers,departures,freight,mail
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,10,2,0,0,0,32,5194000.0,60772.0,842478043.0,84741.0
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,7,10,0,0,0,47,5194000.0,60772.0,842478043.0,84741.0
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,8,8,0,0,0,72,5194000.0,60772.0,842478043.0,84741.0
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,8,3,0,0,0,12,5194000.0,60772.0,842478043.0,84741.0
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,9,0,0,0,0,16,5194000.0,60772.0,842478043.0,84741.0


In [8]:
#Drop non-numeric columns
num_delay_causes = delay_causes_extended.drop(["Date", "UniqueCarrier", "Airline", "TailNum", "Origin", "Org_Airport", "Dest", "Dest_Airport"], axis = 1)
#Drop missing data
num_delay_causes.dropna(axis=0, inplace=True)

target_delay = num_delay_causes["ArrDelay"]
variables_delay = num_delay_causes.drop(["ArrDelay","CarrierDelay", "WeatherDelay",  "NASDelay", "SecurityDelay", "LateAircraftDelay",
                                         "AirTime", "ArrTime","ActualElapsedTime"], axis = 1)

In [29]:
#Scale variables
variables_scl = StandardScaler().fit_transform(variables_delay)

lr_model = LinearRegression()
lr_model.fit(variables_scl,target_delay)
preds = lr_model.predict(variables_scl)
rmse = metrics.root_mean_squared_error(target_delay, preds)

print(f"The RMSE of the model with simple Linear Regression was {rmse:.4f} minutes")

#Create 2nd-degree terms
variables_poly = PolynomialFeatures(degree=2, include_bias=False).fit_transform(variables_scl)

X_train, X_test, y_train, y_test = train_test_split(variables_poly, target_delay, random_state=42)

lr_model_poly = LinearRegression().fit(X_train,y_train)

train_preds = lr_model_poly.predict(X_train)
test_preds = lr_model_poly.predict(X_test)

rmse_poly_train = metrics.root_mean_squared_error(y_train, train_preds)
rmse_poly_test = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE of the model using 2nd degree terms was {rmse_poly_train:.4f} minutes on the training data and {rmse_poly_test:.4f} minutes on the testing data")

The RMSE of the model with simple Linear Regression was 10.7148 minutes
The RMSE of the model using 2nd degree terms was 10.0999 minutes on the training data and 10.0965 minutes on the testing data


In [10]:
print("Variables\t\t\t    Weights")
print("-"*50)

for v,c in zip(variables_delay.columns, lr_model.coef_):
    print(f"{v:<35}{c:.6f}")

Variables			    Weights
--------------------------------------------------
DayOfWeek                          0.000001
DepTime                            0.004924
CRSArrTime                         0.005546
FlightNum                          0.001378
CRSElapsedTime                     0.000626
DepDelay                           0.000094
Distance                           0.004975
TaxiIn                             0.000005
TaxiOut                            0.000009
passengers                         -0.000001
departures                         0.000106
freight                            -0.000000
mail                               -0.000000


The RMSE for linear regression on this dataset was 10.71 minutes. Previously, this model was dominated by the departure delay and taxiing times, but that was on non-standardized data. Simple linear regression is able to somewhat accurately predict flight arrival delays by exploiting the fact that few flights are delayed.

In [30]:
print("Variables\t\t\t     IVF")
print("-"*50)

for i in range(len(variables_delay.columns)):
    vif = variance_inflation_factor(variables_delay, i)
    print(f"{variables_delay.columns[i]:<30}\t{vif:.6f}")

Variables			     IVF
--------------------------------------------------
DayOfWeek                     	4.544551
DepTime                       	24.959457
CRSArrTime                    	25.867108
FlightNum                     	2.537434
CRSElapsedTime                	123.524880
DepDelay                      	2.105158
Distance                      	81.497225
TaxiIn                        	2.549136
TaxiOut                       	2.829834
passengers                    	72.078102
departures                    	77.610892
freight                       	3.088429
mail                          	2.820831


CRSE-Elapsed-Time is the predicted flight time.